# AllasCode — Colab Distillation Pipeline

Este notebook prepara uma baseline de destilação comportamental do teacher Qwen2.5-Coder-7B para um student menor. **Não converte automaticamente a arquitetura em MoE**; veja `MANUAL-DESTILACAO.md` para o caminho de experts + router real.

Linguagem: **TypeScript + Zig (pipeline compartilhado)**. Taxonomia de especialistas idêntica em todos os notebooks.

## 1. Ambiente
Selecione GPU no Colab. Rode primeiro com um subconjunto pequeno. Versões de pacotes podem mudar; se houver incompatibilidade, fixe versões testadas no ambiente de produção.

In [ ]:
!pip -q install -U transformers datasets accelerate peft trl bitsandbytes

In [ ]:
import os, json, random, platform
from pathlib import Path

SEED = 42
random.seed(SEED)

TEACHER_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
# Student menor para a baseline SFT; ajuste conforme VRAM/licença.
STUDENT_MODEL = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
LANGUAGE = "both"
DATA_PATH = "/content/data/train.jsonl"  # JSONL curado e validado
OUTPUT_DIR = "/content/allascode-student"
MAX_LENGTH = 1024

EXPERTS = [
  {
    "id": "syntax_repair",
    "name": "Correção sintática"
  },
  {
    "id": "logic_repair",
    "name": "Correção lógica"
  },
  {
    "id": "memory_safety",
    "name": "Memória e recursos"
  },
  {
    "id": "strict_semantic_typing",
    "name": "Tipagem semântico-nominal estrita e forte"
  },
  {
    "id": "test_engineering",
    "name": "Geração e execução de testes"
  },
  {
    "id": "communication_channels",
    "name": "Canais de comunicação"
  },
  {
    "id": "zero_trust_security",
    "name": "Zero trust e segurança"
  },
  {
    "id": "performance_optimization",
    "name": "Otimização"
  }
]
EXPERT_IDS = {e['id'] for e in EXPERTS}
print(platform.python_version(), LANGUAGE, len(EXPERT_IDS), "canonical experts")

## 2. Contrato do dataset
Cada linha JSONL precisa conter `language`, `expert`, `instruction`, `input` e `output`. O campo expert precisa corresponder exatamente a um ID canônico. Use somente código próprio, sintético ou licenciado. Compile/teste os exemplos antes de treinar.

In [ ]:
def load_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            if not line.strip():
                continue
            row = json.loads(line)
            required = {"language", "expert", "instruction", "input", "output"}
            missing = required - row.keys()
            if missing:
                raise ValueError(f"Linha {line_no}: faltam {sorted(missing)}")
            if row["expert"] not in EXPERT_IDS:
                raise ValueError(f"Linha {line_no}: expert inválido: {row['expert']}")
            if LANGUAGE != "both" and row["language"].lower() not in {LANGUAGE.lower(), "ts" if LANGUAGE == "typescript" else LANGUAGE.lower()}:
                continue
            rows.append(row)
    if not rows:
        raise ValueError("Dataset vazio. Configure DATA_PATH e confirme o filtro de linguagem.")
    return rows

print("DATA_PATH:", DATA_PATH)
print("Crie/envie o JSONL validado antes de seguir. Exemplo de schema:")
print(json.dumps({"language":"typescript","expert":"syntax_repair","instruction":"Corrija e explique.","input":"const n: number = 'x';","output":"..."}, ensure_ascii=False))

## 3. Formatar exemplos para SFT
A resposta-alvo deve incluir solução, justificativa breve e testes relevantes. Não inclua respostas do teacher sem verificação. A etapa abaixo treina no output curado; a geração pelo teacher é uma etapa de preparação de dataset separada.

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from peft import LoraConfig
from trl import SFTTrainer

rows = load_jsonl(DATA_PATH)
# Split por projeto/família de problema antes deste passo é preferível a split aleatório.
random.shuffle(rows)
cut = max(1, int(len(rows) * 0.95))
train_rows, eval_rows = rows[:cut], rows[cut:]
if not eval_rows and len(rows) > 1:
    eval_rows, train_rows = rows[-1:], rows[:-1]

def render(row):
    return {"text": (
        f"<|im_start|>system\nVocê é um especialista em {row['expert']} para {row['language']}. "
        "Priorize correção verificável, explicite incertezas e nunca invente execução de testes.<|im_end|>\n"
        f"<|im_start|>user\nTarefa: {row['instruction']}\nCódigo/contexto:\n{row['input']}<|im_end|>\n"
        f"<|im_start|>assistant\n{row['output']}<|im_end|>"
    )}

train_ds = Dataset.from_list([render(r) for r in train_rows])
eval_ds = Dataset.from_list([render(r) for r in eval_rows]) if eval_rows else None
print("train", len(train_ds), "eval", len(eval_ds) if eval_ds else 0)

## 4. Carregar student e treinar LoRA
Este bloco é um ponto de partida; verifique a compatibilidade entre versões de TRL/Transformers antes de uma execução longa. O modelo student é denso. LoRA reduz parâmetros treináveis, mas não transforma a arquitetura em MoE.

In [ ]:
import torch

bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
tokenizer = AutoTokenizer.from_pretrained(STUDENT_MODEL, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    STUDENT_MODEL,
    torch_dtype=torch.bfloat16 if bf16 else torch.float16,
    device_map="auto",
)
model.config.use_cache = False
peft_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    logging_steps=10,
    save_steps=100,
    eval_strategy="steps" if eval_ds is not None else "no",
    eval_steps=100 if eval_ds is not None else None,
    save_total_limit=2,
    gradient_checkpointing=True,
    bf16=bf16,
    fp16=not bf16,
    report_to="none",
    seed=SEED,
)
trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    peft_config=peft_config,
    processing_class=tokenizer,
    max_seq_length=MAX_LENGTH,
)
# Descomente para iniciar após confirmar dataset, VRAM e configuração:
# trainer.train()
# trainer.save_model(OUTPUT_DIR)
# tokenizer.save_pretrained(OUTPUT_DIR)
print("Preparado. Revise os dados e descomente trainer.train() para iniciar.")

## 5. Validação externa obrigatória
Não use apenas loss como métrica de qualidade. Execute o código gerado em ambiente isolado, sem rede e com limites de recursos. Mantenha um benchmark de teste separado.

In [ ]:
VALIDATION_COMMANDS = {
  "typescript": ["npx tsc --noEmit", "npm test"],
  "zig": ["zig fmt --check .", "zig build test"],
}
print(json.dumps(VALIDATION_COMMANDS, indent=2))
print("Execute apenas os comandos aplicáveis em um checkout sandboxed com toolchain versionada.")

## 6. Próximo passo para MoE real
1. Criar experts distintos com pesos/FFN especializados, mantendo esta taxonomia de IDs.
2. Adicionar router/gating treinável que selecione top-k experts por token ou tarefa.
3. Treinar o router com rótulos de especialista e medir balanceamento de carga, roteamento errado e latência.
4. Comparar contra student denso, teacher e ablação por especialista.
5. Só publicar como MoE quando a arquitetura e o roteamento estiverem presentes e testados.

A destilação de logits exige executar o teacher com acesso a logits e alinhar tokenizers; este notebook faz SFT sobre respostas curadas, não KL-distillation de logits.